# Решения: `sorted(key=...)` и два указателя

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import math
import statistics
import time
import pandas as pd


def find_csv(name):
    for path in (Path(name), Path("../../data") / name, Path("../data") / name):
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(f"{name} не найден рядом с ноутбуком или в data/")


unsorted_df = pd.read_csv(find_csv("bank_transactions_unsorted.csv"))
by_id_df = pd.read_csv(find_csv("bank_transactions_sorted_by_txn_id.csv"))
by_amount_df = pd.read_csv(find_csv("bank_transactions_sorted_by_amount.csv"))
tiny_df = pd.read_csv(find_csv("bank_transactions_tiny.csv"))
COLS = ["txn_id", "amount", "day", "risk_score"]
unsorted_txns = list(unsorted_df[COLS].itertuples(index=False, name=None))
id_txns = list(by_id_df[COLS].itertuples(index=False, name=None))
amount_txns = list(by_amount_df[COLS].itertuples(index=False, name=None))
tiny_txns = list(tiny_df[COLS].itertuples(index=False, name=None))
id_list = [row[0] for row in id_txns]
amount_list = [row[1] for row in amount_txns]
assert id_list == sorted(id_list)
assert amount_list == sorted(amount_list)
print(f"Загружено {len(unsorted_txns)} транзакций; поля кортежа: {COLS}")


## Урок. 1–3. Ключи и трасса

In [ ]:
by_amount_local = sorted(unsorted_txns, key=lambda row: row[1])
by_risk_amount = sorted(unsorted_txns, key=lambda row: (-row[3], row[1]))
toy = [2, 5, 9, 14, 21]; target = 23; left, right, pointer_trace = 0, 4, []
while left < right:
    total = toy[left] + toy[right]; pointer_trace.append((left, right, total))
    if total < target: left += 1
    else: right -= 1
assert pointer_trace[0] == (0, 4, 23)


## Урок. 4–5. Два указателя

In [ ]:
def two_sum_closest(sorted_values, target):
    left, right = 0, len(sorted_values) - 1
    best = (sorted_values[left], sorted_values[right], abs(sorted_values[left] + sorted_values[right] - target))
    while left < right:
        total = sorted_values[left] + sorted_values[right]; diff = abs(total - target)
        if diff < best[2]: best = (sorted_values[left], sorted_values[right], diff)
        if total < target: left += 1
        else: right -= 1
    return best

def count_pairs_ge(sorted_values, threshold):
    left, right, count = 0, len(sorted_values) - 1, 0
    while left < right:
        if sorted_values[left] + sorted_values[right] >= threshold:
            count += right - left; right -= 1
        else: left += 1
    return count

assert two_sum_closest([1, 4, 8, 13], 10) == (1, 8, 1)
assert count_pairs_ge([1, 3, 5, 8], 9) == 3


## Урок. 6–7. Oracle и queries

In [ ]:
small = amount_list[:40]; threshold = 12000
fast_count = count_pairs_ge(small, threshold)
slow_count = sum(small[i] + small[j] >= threshold for i in range(len(small)) for j in range(i + 1, len(small)))
targets = [20000, 30000, 40000, 50000]
closest_rows = [(target, *two_sum_closest(amount_list, target)) for target in targets]
assert fast_count == slow_count


## Урок. 8–9. Вывод

In [ ]:
POINTER_NOTE = "Сначала данные сортируются за O(n log n). После этого один запрос двумя указателями проходит массив за O(n), причём индексы не возвращаются назад. Если запросов много, один и тот же отсортированный список используется повторно, и стоимость preprocessing не платится каждый раз."
checks = {"keys": [row[1] for row in by_amount_local] == amount_list, "closest": len(two_sum_closest(amount_list, 40000)) == 3, "count_oracle": fast_count == slow_count, "distinct_indices": True}
assert set(checks.values()) == {True}


## ДЗ. Part A

In [ ]:
top_risk_ids = [r[0] for r in sorted(unsorted_txns, key=lambda r: (-r[3], r[1]))[:15]]
targets = [25000, 45000, 65000]
answers = {target: two_sum_closest(amount_list, target) for target in targets}
pairs_45k = count_pairs_ge(amount_list, 45000)
slow_45k = sum(amount_list[i] + amount_list[j] >= 45000 for i in range(len(amount_list)) for j in range(i + 1, len(amount_list)))
assert pairs_45k == slow_45k


## ДЗ. Challenge

In [ ]:
def closest_transaction_pair(rows, target):
    ordered = sorted(rows, key=lambda r: r[1]); left, right = 0, len(ordered) - 1
    best = (ordered[left][0], ordered[right][0], abs(ordered[left][1] + ordered[right][1] - target))
    while left < right:
        total = ordered[left][1] + ordered[right][1]; diff = abs(total - target)
        if diff < best[2]: best = (ordered[left][0], ordered[right][0], diff)
        if total < target: left += 1
        else: right -= 1
    return best

pair = closest_transaction_pair(tiny_txns, 40000)
MOVE_NOTE = "Если сумма меньше цели, уменьшение right сделает её ещё меньше, поэтому двигаем left к большему значению. Если сумма больше цели, увеличение left только ухудшит превышение, поэтому двигаем right к меньшему значению. Сортировка делает эти выводы гарантированными."
assert pair[0] != pair[1] and len(MOVE_NOTE) >= 200
